# 📄 arXiv 論文爬蟲
### 領域：電腦視覺 / 圖形辨識（Computer Vision & Image Recognition）
---

## Step 1：import 套件

In [ ]:
import requests                      # 發送 HTTP 請求
import xml.etree.ElementTree as ET   # 解析 XML 格式的回應
import pandas as pd                  # 整理資料成表格
import time                          # 控制爬取速度

## Step 2：定義爬蟲函式

In [ ]:
def fetch_arxiv_papers(query, max_results=10):
    """
    從 arXiv API 爬取論文資料
    
    參數：
        query      : 搜尋關鍵字，例如 'cat:cs.CV AND image recognition'
        max_results: 最多回傳幾筆，預設 10
    
    回傳：
        papers: 論文資料的串列（每篇論文是一個字典）
    """
    
    # arXiv 官方 API 端點
    base_url = "http://export.arxiv.org/api/query"
    
    # 設定查詢參數
    params = {
        "search_query": query,          # 搜尋條件
        "start": 0,                     # 從第 0 筆開始
        "max_results": max_results,     # 最多回傳幾筆
        "sortBy": "submittedDate",      # 依投稿日期排序
        "sortOrder": "descending"       # 最新的在最前面
    }
    
    # 發送 GET 請求
    response = requests.get(base_url, params=params)
    
    # 將回傳的 XML bytes 解析成樹狀結構
    root = ET.fromstring(response.content)
    
    # 宣告命名空間別名（arXiv XML 使用 Atom 格式）
    namespace = {"atom": "http://www.w3.org/2005/Atom"}
    
    papers = []  # 用來存放所有論文資料
    
    # 每篇論文在 XML 中是一個 <entry> 標籤
    for entry in root.findall("atom:entry", namespace):
        paper = {
            # 論文標題，.strip() 去除前後空白
            "title": entry.find("atom:title", namespace).text.strip(),
            
            # 論文摘要（Abstract）
            "summary": entry.find("atom:summary", namespace).text.strip(),
            
            # 投稿日期，[:10] 只取 YYYY-MM-DD
            "published": entry.find("atom:published", namespace).text[:10],
            
            # 作者可能有多位，用 list comprehension 收集成串列
            "authors": [
                author.find("atom:name", namespace).text
                for author in entry.findall("atom:author", namespace)
            ],
            
            # 論文的 arXiv 連結
            "link": entry.find("atom:id", namespace).text
        }
        papers.append(paper)
    
    return papers

## Step 3：執行爬蟲，搜尋電腦視覺論文

In [ ]:
# 搜尋電腦視覺分類下，與圖形辨識相關的論文
papers = fetch_arxiv_papers(
    query="cat:cs.CV AND (image recognition OR object detection)",
    max_results=10
)

print(f"✅ 共爬取到 {len(papers)} 篇論文")

## Step 4：轉成 DataFrame 觀看結果

In [ ]:
# 將串列轉成 DataFrame
df = pd.DataFrame(papers)

# 只顯示標題和日期
print(df[["title", "published"]].to_string())

In [ ]:
# 查看完整 DataFrame（在 Jupyter 中會顯示成漂亮的表格）
df

## Step 5：查看單篇論文的詳細資訊

In [ ]:
# 查看第一篇論文的完整內容
first_paper = papers[0]

print("📌 標題：", first_paper["title"])
print("📅 日期：", first_paper["published"])
print("👤 作者：", ", ".join(first_paper["authors"]))
print("🔗 連結：", first_paper["link"])
print("\n📝 摘要：")
print(first_paper["summary"])

## Step 6：儲存成 CSV 檔案

In [ ]:
# 儲存成 CSV（utf-8-sig 讓 Excel 開啟時中文不會亂碼）
df.to_csv("cv_papers.csv", index=False, encoding="utf-8-sig")

print(f"✅ 已儲存！共 {len(df)} 筆論文 → cv_papers.csv")

---
## ⚠️ 注意：大量爬取時請加延遲

In [ ]:
# 如果要爬多個關鍵字，每次之間要休息，避免對伺服器造成負擔

queries = [
    "cat:cs.CV AND image recognition",
    "cat:cs.CV AND object detection",
    "cat:cs.CV AND image segmentation"
]

all_papers = []

for q in queries:
    print(f"🔍 搜尋中：{q}")
    result = fetch_arxiv_papers(query=q, max_results=5)
    all_papers.extend(result)
    time.sleep(3)  # 每次請求之間等待 3 秒

df_all = pd.DataFrame(all_papers)

# 去除重複論文（同一篇可能被多個關鍵字搜到）
df_all = df_all.drop_duplicates(subset="link")

print(f"\n✅ 總共爬取到 {len(df_all)} 篇不重複論文")
df_all[["title", "published"]]